# External frozen inference

**Scientific purpose.** Document the cohort-specific external adapter and inspect verified
aggregate HCC sensitivities without training or recalibration.

**Runnability classification:** aggregate reanalysis is runnable from a clean clone.
Complete historical CNN inference is not reconstructable because W3 fold 1 and W4 fold 0
checkpoints were not retained. Any authorized private rerun must use frozen artifacts only.

**Inputs:** released aggregate external table; private adapters/models for partial workflow
inspection. **Outputs:** aggregate HCC-sensitivity review only.

The 103 cases are all known HCC. This is an HCC-recognition transportability stress test,
not multiclass external validation, and no external multiclass AUC is reported.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy.ndimage import binary_dilation

from liver_tumor_pipeline.constants import FULL_FUSION_FEATURES
from liver_tumor_pipeline.external import compare_sex_scenarios

PHASE_ORDER = ("P", "C1", "C2", "C3")
TARGET_SPACING = (1.0, 1.0, 1.0)
CROP_SHAPE = (96, 96, 96)


## External adapter preprocessing

Unlike the historical internal pipeline, the external adapter resamples available inputs
to 1-mm spacing, applies a liver window for crop-restricted radiomics, uses a shared tumor-
union centroid within the external patient, and leaves unavailable phase channels empty.
A segmented liver is preferred; a 15-mm tumor dilation is the fallback. These deviations
are part of the stress test and are not presented as internal preprocessing.


In [ ]:
def resample_array(array, spacing, *, is_mask: bool):
    image = sitk.GetImageFromArray(array)
    image.SetSpacing(tuple(spacing))
    new_size = [
        int(round(size * old / new))
        for size, old, new in zip(image.GetSize(), image.GetSpacing(), TARGET_SPACING)
    ]
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(TARGET_SPACING)
    resampler.SetSize(new_size)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0 if is_mask else -1024)
    output = sitk.GetArrayFromImage(resampler.Execute(image))
    return (output > 0).astype(np.uint8) if is_mask else output.astype(np.float32)


def liver_window(volume, width=150.0, center=50.0):
    lower, upper = center - width / 2, center + width / 2
    return ((np.clip(volume, lower, upper) - lower) * 255.0 / (upper - lower)).astype(np.float32)


def crop_at_center(volume, center, shape=CROP_SHAPE):
    output = np.zeros(shape, dtype=volume.dtype)
    starts = [int(value) - size // 2 for value, size in zip(center, shape)]
    ends = [start + size for start, size in zip(starts, shape)]
    source = tuple(
        slice(max(0, start), min(volume.shape[axis], end))
        for axis, (start, end) in enumerate(zip(starts, ends))
    )
    destination_starts = [max(0, -start) for start in starts]
    source_shape = volume[source].shape
    destination = tuple(
        slice(start, start + length)
        for start, length in zip(destination_starts, source_shape)
    )
    output[destination] = volume[source]
    return output


def normalize_nonzero(volume):
    output = np.zeros_like(volume, dtype=np.float32)
    mask = volume > 0
    if mask.sum() < 100:
        return volume.astype(np.float32)
    values = volume[mask]
    output[mask] = (values - values.mean()) / (values.std() + 1e-6)
    return output


def preprocess_external_record(record):
    images, tumors, livers = {}, {}, {}
    for phase in PHASE_ORDER:
        image = record["ct_by_phase"].get(phase)
        tumor = record["tumor_by_phase"].get(phase)
        liver = record.get("liver_by_phase", {}).get(phase)
        images[phase] = None if image is None else liver_window(
            resample_array(image, record["spacing_by_phase"][phase], is_mask=False)
        )
        tumors[phase] = None if tumor is None else resample_array(
            tumor, record["spacing_by_phase"][phase], is_mask=True
        )
        livers[phase] = None if liver is None else resample_array(
            liver, record["spacing_by_phase"][phase], is_mask=True
        )
    available_masks = [mask for mask in tumors.values() if mask is not None]
    if not available_masks:
        raise ValueError("External record has no usable tumor mask.")
    reference_shape = available_masks[0].shape
    union = np.zeros(reference_shape, dtype=np.uint8)
    for mask in available_masks:
        if mask.shape == reference_shape:
            union = np.maximum(union, mask)
    if union.sum() == 0:
        raise ValueError("External tumor-union mask is empty.")
    center = tuple(np.argwhere(union > 0).mean(axis=0).astype(int))
    liver_candidates = [
        livers[phase] for phase in ("C2", "C1", "P", "C3")
        if livers[phase] is not None and livers[phase].shape == reference_shape
    ]
    liver = liver_candidates[0] if liver_candidates else binary_dilation(
        union > 0, iterations=15
    ).astype(np.uint8)
    cnn = np.zeros((4, *CROP_SHAPE), dtype=np.float32)
    radiomics = np.zeros((4, *CROP_SHAPE), dtype=np.float32)
    available_phases = []
    for channel, phase in enumerate(PHASE_ORDER):
        image = images[phase]
        if image is None or image.shape != reference_shape:
            continue
        cropped = crop_at_center(image * (liver > 0), center)
        radiomics[channel] = cropped
        cnn[channel] = normalize_nonzero(cropped)
        available_phases.append(phase)
    return {"cnn": cnn, "radiomics": radiomics, "available_phases": available_phases}


## Frozen-model and full-fusion boundaries

W3 and W4 probabilities were averaged across available frozen fold models. Full fusion
used exactly W4 and W5 three-class probabilities, age, and sex. W3 was excluded. External
age was replaced by the locked fold-specific internal median. Historical code encoded
unavailable sex as 0; a deterministic sensitivity analysis substituted the locked fold
median sex of 1 without refitting or recalibration.

No verified external CNN-plus-radiomics-only artifact exists, so that result must not be
reported.


In [ ]:
aggregate_path = REPO_ROOT / "results" / "aggregate" / "external_stress_test.csv"
if not aggregate_path.is_file():
    raise FileNotFoundError("Released aggregate external stress-test table is unavailable.")
external_results = pd.read_csv(aggregate_path)
external_results


## Verified interpretation

W3 recognized 5/103 HCC cases (0.049); W4 recognized 0/103 (0.000). The historical
full-fusion adapter with sex=0 recognized 2/103 (0.019). With sex=1 fixed to the locked
fold median, 83/103 were recognized (0.806), and predicted labels changed for 82/103.

The sex=1 result is a sensitivity scenario, not the originally implemented result and not
evidence that every external patient's sex was 1. The instability prevents a claim of
robust external fusion transportability.
